#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("schemaBronze", "bronze")
dbutils.widgets.text("schemaSilver", "silver")
dbutils.widgets.text("catalogo", "project_dev")

In [0]:
schemaBronze = dbutils.widgets.get("schemaBronze")
schemaSilver = dbutils.widgets.get("schemaSilver")
catalogo = dbutils.widgets.get("catalogo")

# definir rutas
path_inicio = f"{catalogo}.{schemaBronze}"
path_target = f"{catalogo}.{schemaSilver}"

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, count, col, round, sum, when, lit
from pyspark.sql import functions as F

##orders items

In [0]:

df_orders_items = spark.table( f"{catalogo}.{schemaBronze}.orders_items ")

In [0]:
# se aplica una columna row_number para la columna order_id para contar las veces que aoparezca el order_id
windowSpec = Window.partitionBy("order_id").orderBy("order_id")

df_orders_items = (
    df_orders_items.withColumn( "row_order", count("order_id").over(windowSpec) )
    .withColumn("total", col("price") + col("freight_value"))
    .withColumn("porcentaje_price",  round(col("price") / col("total"),4) )
    .withColumn("porcentaje_freight",  round(col("freight_value") / col("total"), 4))
)


In [0]:
### SELECCIONA SOLO LAS COLUMNAS NECESARIAS, ELIMINA DUPLIUCADOS YU CREA UN NUEVO DATAFRAME

df_orders_items_distinct = df_orders_items.select("order_id", "row_order", "total", "porcentaje_price", "porcentaje_freight").distinct().withColumnRenamed("order_id", "order_id_items")

##Orders payment

In [0]:
df_orders_payments = (
    spark.table( f" {catalogo}.{schemaBronze}.orders_payments ")
    .select( 
            F.col("order_id").alias("order_id_pay"),
            "payment_type", 
            "payment_value"
    )
)

# crear la lista con los uniques de payment_type
type_pay = [row["payment_type"] for row in df_orders_payments.select("payment_type").distinct().collect()]
#df_orders_payments.limit(5).display()

In [0]:
df_payments_group = (
    df_orders_payments
    .select("order_id_pay", "payment_type", "payment_value")
    .groupBy("order_id_pay", "payment_type")
    .agg(sum("payment_value").alias("payment_value"))
)

In [0]:
df_payments_group = (
    df_payments_group.alias("p").join( df_orders_items_distinct.alias("i") , df_payments_group["order_id_pay"] == df_orders_items_distinct["order_id_items"],how="inner")
    .drop("order_id_items")
    .withColumn("payment_value_div", round( col("p.payment_value") / col("i.row_order"), 3) )
    .withColumn("price_end", round( col("payment_value_div") * col("i.porcentaje_price"), 3) )
    .withColumn("freight_end", round( col("payment_value_div") * col("i.porcentaje_freight"), 3))
)
#df_payments_group.limit(10).display()

In [0]:
### AGRUOPA POR ORDER Y PAYMENT, Y HACE UN PIVOT POR CADA TIPO DE PAGO

df_payments_pivot = (
    df_payments_group
    .select("order_id_pay", "payment_type", "price_end", "freight_end", "total")
    .groupBy("order_id_pay", "total")
    .pivot("payment_type", type_pay)
    .agg(
        F.sum("price_end").alias("price_end"),
        F.sum("freight_end").alias("freight_end")
    )
    .fillna(0)
)

In [0]:
'''df_prueba = (
    df_payments_group
    .select("order_id_pay", "payment_type", "price_end", "freight_end", "total")
    .groupBy("order_id_pay", "total")
    .pivot("payment_type", type_pay)
    .agg(
        F.sum("price_end").alias("price_end"),
        F.sum("freight_end").alias("freight_end")
    )
    .fillna(0)
)

df_prueba[df_prueba['order_id_pay'] == '014405982914c2cde2796ddcf0b8703d'].display()

df_payments_pivot = (
    df_payments_group
    .select("order_id_pay", "payment_type", "price_end", "freight_end")
    .groupBy("order_id_pay", "payment_type")
    .agg(
        F.sum("price_end").alias("price_end_"),
        F.sum("freight_end").alias("freight_end_")
    )
    .groupBy("order_id_pay")
    .pivot("payment_type", type_pay)
    .agg(
        F.sum("price_end_").alias("price_end"),
        F.sum("freight_end_").alias("freight_end")
    )
    .fillna(0)
)'''

## orders dataset

In [0]:
### leer el dataframe
df_orders = (
    spark.table( f" {catalogo}.{schemaBronze}.orders")
    .select( 
            F.col("order_id").alias("order_id_stt"),
            "customer_id", 
            "order_status",
            "order_approved_at",
            # col5
            when( col( "order_delivered_carrier_date").isNull(), col( "order_approved_at"))
            .otherwise( col("order_delivered_carrier_date"))
            .alias("order_delivered_carrier_date")
    )
)

## join items, payments

In [0]:
### realizar el join con las dos tablas
df_orders_items = (
    df_orders_items.alias("i").join( df_payments_pivot.alias("p"),
                                    (df_orders_items["order_id"] == df_payments_pivot["order_id_pay"]) & 
                                    (df_orders_items["total"] == df_payments_pivot["total"]),
                                    how="inner")
    .join( df_orders.alias("o") , df_orders_items["order_id"] == df_orders["order_id_stt"],how="inner")
    .drop("price", "freight_value", "row_order", "total", "porcentaje_price", "porcentaje_freight", "order_id_pay", "order_id_stt")
)

#df_orders_items.display()

In [0]:
### guardar la tabla clean en el target
df_orders_items.write.mode("overwrite").insertInto(f"{path_target}.orders_transform")

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${schemaSilver}.orders_transform
ZORDER BY product_id, seller_id

In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${schemaSilver}.orders_transform RETAIN 24 HOURS

In [0]:
%sql
--- describir la historia de la tabla
DESCRIBE HISTORY ${catalogo}.${schemaSilver}.orders_transform